In [1]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [2]:
# ============================================================
# 1. MEMBUAT DATASET PELANGGAN (SINTETIS)
# ============================================================
print("="*60)
print("DATASET PELANGGAN (SINTETIS)")
print("="*60)

np.random.seed(42)
n = 200

income = np.concatenate([
    np.random.normal(30, 5, 50),
    np.random.normal(30, 5, 50),
    np.random.normal(70, 8, 50),
    np.random.normal(70, 8, 50),
])

spending = np.concatenate([
    np.random.normal(20, 5, 50),
    np.random.normal(70, 5, 50),
    np.random.normal(20, 5, 50),
    np.random.normal(70, 5, 50),
])

X = np.column_stack([income, spending])
df = pd.DataFrame(X, columns=["Income", "Spending"])

print(df.head())

DATASET PELANGGAN (SINTETIS)
      Income   Spending
0  32.483571  21.788937
1  29.308678  22.803923
2  33.238443  25.415256
3  37.615149  25.269010
4  28.829233  13.111653


In [3]:
# ============================================================
# 2. MENENTUKAN K OPTIMAL (ELBOW + SILHOUETTE)
# ============================================================
print("\n" + "="*60)
print("MENENTUKAN K OPTIMAL")
print("="*60)

inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))
    print(f"K={k} → Inertia={km.inertia_:.1f}  Silhouette={silhouette_score(X, labels):.3f}")

best_k = list(K_range)[np.argmax(silhouettes)]
print(f"\nK optimal berdasarkan silhouette tertinggi: {best_k}")

# Plot Elbow + Silhouette
fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].plot(list(K_range), inertias, "bo-")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(K_range), silhouettes, "ro-")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")

plt.tight_layout()
plt.savefig("pertemuan07_elbow_silhouette.png")
plt.close()



MENENTUKAN K OPTIMAL
K=2 → Inertia=137362.8  Silhouette=0.422
K=3 → Inertia=54205.9  Silhouette=0.623
K=4 → Inertia=12554.4  Silhouette=0.753
K=5 → Inertia=10927.6  Silhouette=0.662
K=6 → Inertia=9806.1  Silhouette=0.560
K=7 → Inertia=8882.7  Silhouette=0.468
K=8 → Inertia=7397.2  Silhouette=0.489
K=9 → Inertia=6877.0  Silhouette=0.470
K=10 → Inertia=6199.5  Silhouette=0.472

K optimal berdasarkan silhouette tertinggi: 4


In [4]:
# ============================================================
# 3. K-MEANS CLUSTERING
# ============================================================
print("\n" + "="*60)
print("K-MEANS CLUSTERING")
print("="*60)

kmeans = KMeans(n_clusters=best_k, random_state=42)
labels_k = kmeans.fit_predict(X)
df["Cluster"] = labels_k

print("Silhouette Score:", round(silhouette_score(X, labels_k), 3))

# ============================================================
# 4. PROFIL TIAP CLUSTER
# ============================================================
print("\nPROFIL CLUSTER")
print(f"{'Cluster':>8}  {'Avg Income':>12}  {'Avg Spending':>13}  {'Count':>6}")

profiles = df.groupby("Cluster").agg({
    "Income":"mean",
    "Spending":"mean",
    "Cluster":"count"
}).rename(columns={"Cluster":"Count"})

for idx, row in profiles.iterrows():
    print(f"{idx:>8d}  {row['Income']:>12.1f}  {row['Spending']:>13.1f}  {int(row['Count']):>6d}")


K-MEANS CLUSTERING
Silhouette Score: 0.753

PROFIL CLUSTER
 Cluster    Avg Income   Avg Spending   Count
       0          30.1           69.9      50
       1          70.7           70.9      50
       2          28.9           20.8      50
       3          69.7           20.2      50


In [5]:
# ============================================================
# 5. PEMBERIAN NAMA SEGMEN
# ============================================================
print("\nNAMA SEGMEN")

segment_names = {}

for idx, row in profiles.iterrows():
    inc = row["Income"]
    sp = row["Spending"]

    if inc > 50 and sp > 50:
        name = "High Value"
    elif inc > 50 and sp <= 50:
        name = "Conservative"
    elif inc <= 50 and sp > 50:
        name = "Impulsive"
    else:
        name = "Budget"

    segment_names[idx] = name
    print(f"Cluster {idx}: {name} (Income={inc:.0f}, Spending={sp:.0f})")

df["Segment"] = df["Cluster"].map(segment_names)

# ============================================================
# 6. VISUALISASI CLUSTER
# ============================================================
plt.figure(figsize=(7,5))
for seg in df["Segment"].unique():
    mask = df["Segment"] == seg
    plt.scatter(df[mask]["Income"], df[mask]["Spending"], label=seg, alpha=0.7)

plt.xlabel("Income")
plt.ylabel("Spending")
plt.title("Segmentasi Pelanggan (K-Means)")
plt.legend()
plt.tight_layout()
plt.savefig("pertemuan07_segmentasi.png")
plt.close()

print("\nGrafik disimpan:")
print("- pertemuan07_elbow_silhouette.png")
print("- pertemuan07_segmentasi.png")



NAMA SEGMEN
Cluster 0: Impulsive (Income=30, Spending=70)
Cluster 1: High Value (Income=71, Spending=71)
Cluster 2: Budget (Income=29, Spending=21)
Cluster 3: Conservative (Income=70, Spending=20)

Grafik disimpan:
- pertemuan07_elbow_silhouette.png
- pertemuan07_segmentasi.png


In [6]:
# ============================================================
# 7. HIERARCHICAL CLUSTERING + DENDROGRAM
# ============================================================
print("\nHIERARCHICAL CLUSTERING")

linked = linkage(X, method='ward')

plt.figure(figsize=(8,5))
dendrogram(linked)
plt.title("Dendrogram Hierarchical Clustering")
plt.tight_layout()
plt.savefig("pertemuan07_dendrogram.png")
plt.close()

print("Dendrogram disimpan ke pertemuan07_dendrogram.png")



HIERARCHICAL CLUSTERING
Dendrogram disimpan ke pertemuan07_dendrogram.png


In [7]:
# ============================================================
# 8. 3 REKOMENDASI BISNIS
# ============================================================
print("\n" + "="*60)
print("3 REKOMENDASI BISNIS")
print("="*60)

print("1. Segmen 'High Value':")
print("   Fokus pada loyalty program, layanan premium, dan personalisasi eksklusif.")
print("   Pertahankan retensi karena kontribusi revenue tinggi.")

print("\n2. Segmen 'Impulsive':")
print("   Gunakan flash sale, limited offer, dan promosi berbasis urgensi.")
print("   Mereka responsif terhadap stimulus diskon.")

print("\n3. Segmen 'Conservative' & 'Budget':")
print("   Tawarkan paket hemat dan diskon bertahap untuk meningkatkan volume transaksi.")
print("   Edukasi produk untuk meningkatkan willingness to spend.")

print("\n✅ Pertemuan 07 selesai.")


3 REKOMENDASI BISNIS
1. Segmen 'High Value':
   Fokus pada loyalty program, layanan premium, dan personalisasi eksklusif.
   Pertahankan retensi karena kontribusi revenue tinggi.

2. Segmen 'Impulsive':
   Gunakan flash sale, limited offer, dan promosi berbasis urgensi.
   Mereka responsif terhadap stimulus diskon.

3. Segmen 'Conservative' & 'Budget':
   Tawarkan paket hemat dan diskon bertahap untuk meningkatkan volume transaksi.
   Edukasi produk untuk meningkatkan willingness to spend.

✅ Pertemuan 07 selesai.
